# Draft-Revise: Systematic Study

The most robust CTM enhancement. This notebook provides a publication-grade
experimental validation with three tiers:

| tier | what | runs |
|---|---|---|
| **Main** | baseline vs revise (w=0.1, cp=0.15), 5 seeds x 4 tasks | 20 |
| **Sweep** | weight {0.05,0.1,0.2,0.3} x corrupt {0.05,0.15,0.3}, 3 seeds x 2 tasks | 72 |
| **Ablation** | full / no-noise / no-revise-loss / block1 / block3, 3 seeds x 2 tasks | 30 |
| **Total** | | **122** |

**Hardware**: 1 machine x 8 GPUs (~16h).

In [ ]:
import sys; sys.path.insert(0, '.'); sys.path.insert(0, '..')
from exp_runner import *
import matplotlib.pyplot as plt
%matplotlib inline

## Part A - Prior Results (st10)

How draft-revise performed in the initial sweep.

In [ ]:
df_prior = load_prior()
if df_prior is not None:
    print(summary_stats(df_prior[df_prior.stage == 'st10']))
else:
    print('Prior data not found in csv_data/')

In [ ]:
if df_prior is not None:
    plot_prior_bar(df_prior, ['cifar10','mazes','parity','sort'],
                   'st10', 'revise', 'Prior: draft-revise vs baseline',
                   'figures/01_prior_bar.png')

In [ ]:
curves = load_prior_curves()
if curves:
    plot_prior_curves(curves, 'sort',
        [('st00','paper','baseline','#888'),
         ('st10','revise','revise','#9467bd')],
        'Sort convergence (prior)', 'figures/01_prior_conv.png')

## Part B - Experiment Design

Three tiers: main validation, hyperparameter sweep, and ablation.

In [ ]:
# Tier 1: Main (5 seeds x 4 tasks)
main_exps = make_revise(['cifar10','mazes','sort','parity'], [0,1,2,3,4], w=0.1, cp=0.15)
# Tier 2: Sweep (weight x corrupt_prob, 3 seeds x 2 tasks)
sweep_exps = make_revise_sweep(['sort','mazes'], [0,1,2])
# Tier 3: Ablation (5 variants, 3 seeds x 2 tasks)
ablation_exps = make_revise_ablation(['sort','mazes'], [0,1,2])

exps = main_exps + sweep_exps + ablation_exps
print(f'Main: {len(main_exps)}, Sweep: {len(sweep_exps)}, Ablation: {len(ablation_exps)}')
print(f'Total: {len(exps)} experiments')

In [ ]:
# Dry run (shows first 5 + last 5 commands)
run_all(exps[:5], gpus=8, log_root='logs/deep/01_revise', dry_run=True)
print('...')
run_all(exps[-5:], gpus=8, log_root='logs/deep/01_revise', dry_run=True)

### Run all (~16h on 8 GPUs)

In [ ]:
done, failed = run_all(exps, gpus=8, log_root='logs/deep/01_revise')

In [ ]:
status('logs/deep/01_revise')

## Part C - Main Results

Baseline vs draft-revise with 5-seed statistical significance.

In [ ]:
df = collect('logs/deep/01_revise')
if df.empty:
    print('No results yet.')
else:
    df_main = df[df.name.str.contains('revise_w0p1_cp0p15') & ~df.name.str.contains('swp|abl')]
    if not df_main.empty:
        print(df_main[['name','task','best_acc','delta']].to_string(index=False))
        plot_delta_bars(df_main, 'Main: revise vs baseline (5 seeds)',
                        'figures/01_main_delta.png')

In [ ]:
if not df.empty:
    df_main = df[df.name.str.contains('revise_w0p1_cp0p15') & ~df.name.str.contains('swp|abl')]
    if not df_main.empty:
        sig = significance_test(df_main)
        print(sig.to_string(index=False))

In [ ]:
if not df.empty:
    df_main = df[df.name.str.contains('revise_w0p1_cp0p15') & ~df.name.str.contains('swp|abl')]
    if not df_main.empty:
        plot_box_seeds(df_main, 'task', 'best_acc',
                       'Main: seed variance (5 seeds/task)',
                       'figures/01_main_box.png')

## Part D - Hyperparameter Sensitivity

2D sweep: revise_weight x corrupt_prob, shown as delta-vs-baseline heatmap.

In [ ]:
if not df.empty:
    import re
    df_sw = df[df.name.str.contains('swp_')].copy()
    if not df_sw.empty:
        df_sw['weight'] = df_sw['name'].str.extract(r'w([0-9]+p?[0-9]*)_cp')[0].str.replace('p','.').astype(float)
        df_sw['corrupt'] = df_sw['name'].str.extract(r'cp([0-9]+p?[0-9]*)_s')[0].str.replace('p','.').astype(float)
        df_sw['variant'] = df_sw['weight'].astype(str) + '/' + df_sw['corrupt'].astype(str)
        plot_sweep_heatmap(df_sw, 'weight', 'task',
                          'Revise: weight x task delta (pp)',
                          'figures/01_sweep_heatmap.png')

In [ ]:
if not df.empty:
    df_sw = df[df.name.str.contains('swp_')].copy()
    if not df_sw.empty:
        import re
        df_sw['weight'] = df_sw['name'].str.extract(r'w([0-9]+p?[0-9]*)_cp')[0].str.replace('p','.').astype(float)
        plot_sweep_curve(df_sw, 'weight',
                        'Revise weight sweep (corrupt=0.15 highlighted)',
                        'figures/01_sweep_curve.png')

## Part E - Ablation Study

What drives the improvement? Each variant removes or changes one component.

In [ ]:
if not df.empty:
    df_abl = df[df.name.str.contains('abl_')].copy()
    if not df_abl.empty:
        import re
        df_abl['variant'] = df_abl['name'].str.extract(r'abl_([a-z_]+)_s')[0]
        plot_ablation_bars(df_abl, 'variant',
                          'Revise ablation: component contributions',
                          'figures/01_ablation.png')

In [ ]:
if not df.empty:
    df_abl = df[df.name.str.contains('abl_')].copy()
    if not df_abl.empty:
        import re
        df_abl['variant'] = df_abl['name'].str.extract(r'abl_([a-z_]+)_s')[0]
        print(summary_stats(df_abl, groupby=('task','variant')))